# Notebook 10: simulation study

Controlled low-order simulation of common-component confounding and network-effect recovery. The fitted specifications are the raw GNAR(2, [1, 1]) and the same model after covariance-PC1 adjustment.

In [ ]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *

import time
import numpy as np
import pandas as pd
from scipy import stats as _st

QUICK = quick_mode()

N_REPS = 200 if not QUICK else 10
DENSITY_REPS = max(N_REPS // 2, 5)
T_SIM = 300
BURN_IN = 300
N_SIM = 23

# Fixed low-order simulation calibration. The network coefficients are rounded so
# that the planted lag-sum effect is exactly 0.015.
CAL = dict(
    a1=1.1277,
    a2=-0.1431,
    b1=0.060,
    b2=-0.045,
    lam=0.020,
    phi_f=0.977,
    sig_e=0.37,
)

print("calibration:", CAL)
print(f"principal replications={N_REPS}; density-ladder replications={DENSITY_REPS}")

calibration: {'a1': 1.1277, 'a2': -0.1431, 'b1': 0.06, 'b2': -0.045, 'lam': 0.02, 'phi_f': 0.977, 'sig_e': 0.37}
principal replications=10; density-ladder replications=5


## Synthetic topology and data-generating process

The principal simulation uses the fixed four-community graph. All simulations start from zero, use a 300-period warm-up and retain the following 300 observations.

In [2]:
def block_W(n=N_SIM, n_blocks=4, proposal_p_in=0.7, proposal_p_out=0.05, seed=0):
    """Generate the row-normalised synthetic four-community graph."""
    rng = np.random.default_rng(seed)
    block = np.arange(n) % n_blocks
    adjacency = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            p = proposal_p_in if block[i] == block[j] else proposal_p_out
            adjacency[i, j] = rng.random() < p

    adjacency = ((adjacency + adjacency.T) > 0).astype(float)
    np.fill_diagonal(adjacency, 0.0)

    row_sums = adjacency.sum(axis=1, keepdims=True)
    if np.any(row_sums == 0):
        raise RuntimeError("Synthetic block graph contains an isolated node.")
    return adjacency / row_sums


def simulate_panel(
    T, n, a1, a2, b1, b2, lam, W,
    phi_f=0.977, sig_e=0.37, burn=BURN_IN, seed=0, loadings=None,
):
    """Generate one panel from the fixed simulation DGP."""
    rng = np.random.default_rng(seed)
    total = T + burn
    Y = np.zeros((total, n))
    factor = 0.0

    loading_vec = np.ones(n) if loadings is None else np.asarray(loadings, dtype=float)
    if loading_vec.shape != (n,):
        raise ValueError("loadings must have length n.")

    use_network = b1 != 0.0 or b2 != 0.0

    for t in range(2, total):
        factor = phi_f * factor + rng.standard_normal()
        net1 = W @ Y[t - 1] if use_network else 0.0
        net2 = W @ Y[t - 2] if use_network else 0.0

        Y[t] = (
            a1 * Y[t - 1]
            + a2 * Y[t - 2]
            + b1 * net1
            + b2 * net2
            + lam * loading_vec * factor
            + sig_e * rng.standard_normal(n)
        )

    return Y[burn:]


W_blk = block_W()
adjacency = W_blk > 0
np.fill_diagonal(adjacency, False)

block_density = float(adjacency.sum() / (N_SIM * (N_SIM - 1)))
rad = full_system_radius(
    CAL["a1"], CAL["a2"], CAL["b1"], CAL["b2"], W_blk
)
dom = float(max(abs(np.roots([1, -CAL["a1"], -CAL["a2"]]))))

print(
    f"block graph density={block_density:.5f}; "
    f"mean degree={adjacency.sum(1).mean():.5g}"
)
print(f"domestic-only AR(2) root={dom:.5f}")
print(f"full-system spectral radius={rad:.5f}")
assert rad < 1

block graph density=0.25296; mean degree=5.5652
domestic-only AR(2) root=0.98197
full-system spectral radius=0.99951


## Common-component calibration

The empirical covariance-PC1 variance share is used only as a calibration target for the supplementary factor-strength and geographic-densification checks.

In [3]:
PC1_TARGET = float(load_result("nb6_common_factor")["pc1_share"])


def realised_share(lam, b1, b2, seeds=range(5), loadings=None, W=W_blk):
    """Mean realised covariance-PC1 variance share for a fixed calibration."""
    shares = []

    for seed in seeds:
        Y = simulate_panel(
            T_SIM,
            N_SIM,
            CAL["a1"],
            CAL["a2"],
            b1,
            b2,
            lam,
            W,
            seed=seed,
            sig_e=CAL["sig_e"],
            loadings=loadings,
        )

        if not np.all(np.isfinite(Y)) or np.abs(Y).max() > 1e6:
            return np.nan

        centred = Y - Y.mean(axis=0, keepdims=True)
        singular = np.linalg.svd(centred, compute_uv=False)
        shares.append(singular[0] ** 2 / np.sum(singular ** 2))

    return float(np.mean(shares))


def match_lam(
    loadings,
    b1,
    b2,
    target=PC1_TARGET,
    W=W_blk,
    hi_start=0.05,
    hi_max=32.0,
    iterations=24,
    tolerance=5e-4,
    strict=True,
):
    """Calibrate factor strength to a target realised covariance-PC1 share."""
    loadings = np.asarray(loadings, dtype=float)

    lo = 0.0
    share_lo = realised_share(lo, b1, b2, loadings=loadings, W=W)
    if share_lo > target:
        msg = (
            f"PC1 target {target:.5f} is below the zero-loading share "
            f"{share_lo:.5f}."
        )
        if strict:
            raise RuntimeError(msg)
        return lo, share_lo, False

    hi = hi_start
    share_hi = realised_share(hi, b1, b2, loadings=loadings, W=W)
    last_finite_hi, last_finite_share = lo, share_lo

    while np.isfinite(share_hi) and share_hi < target and hi < hi_max:
        last_finite_hi, last_finite_share = hi, share_hi
        hi *= 2.0
        share_hi = realised_share(hi, b1, b2, loadings=loadings, W=W)

    if not np.isfinite(share_hi) or share_hi < target:
        msg = (
            f"Could not bracket PC1 target {target:.5f}; largest finite attempt "
            f"was loading {last_finite_hi:.5f} with share "
            f"{last_finite_share:.5f}."
        )
        if strict:
            raise RuntimeError(msg)
        return float(last_finite_hi), float(last_finite_share), False

    for _ in range(iterations):
        mid = 0.5 * (lo + hi)
        share_mid = realised_share(mid, b1, b2, loadings=loadings, W=W)
        if share_mid < target:
            lo = mid
        else:
            hi = mid

    lam = 0.5 * (lo + hi)
    share = realised_share(lam, b1, b2, loadings=loadings, W=W)
    ok = abs(share - target) <= tolerance

    if strict and not ok:
        raise RuntimeError(
            f"Factor calibration missed target: {share:.5f} versus {target:.5f}."
        )

    return float(lam), float(share), bool(ok)


realised_shares = {}
for label, lam, b1, b2 in [
    ("DGP1 AR only", 0.0, 0.0, 0.0),
    ("DGP2 AR+factor", CAL["lam"], 0.0, 0.0),
    ("DGP3 AR+net", 0.0, CAL["b1"], CAL["b2"]),
    ("DGP4 AR+factor+net", CAL["lam"], CAL["b1"], CAL["b2"]),
]:
    realised_shares[label] = realised_share(lam, b1, b2)
    print(f"{label:22s} PC1 share={realised_shares[label]:.5f}")

print(f"empirical covariance-PC1 share={PC1_TARGET:.5f}")

DGP1 AR only           PC1 share=0.33837
DGP2 AR+factor         PC1 share=0.76973
DGP3 AR+net            PC1 share=0.37205
DGP4 AR+factor+net     PC1 share=0.87715
empirical covariance-PC1 share=0.88829


## Raw and PC1-adjusted GNAR fits

The two fitted specifications differ only by the covariance-PC1 adjustment. The PC1 loading vector is estimated separately from each simulated panel and removed before the same conjugate GNAR is fitted.

In [4]:
SPECS = ["raw", "pc1_adjusted"]


def _coef_names(p=2, s=(1, 1)):
    """Column order used by BayesianGNAR."""
    names = [f"own_lag{j + 1}" for j in range(p)]
    for j in range(p):
        for r in range(s[j]):
            names.append(f"net_lag{j + 1}_stage{r + 1}")
    return names


def _net_sum_interval(model, p=2, s=(1, 1), level=0.95):
    """Posterior mean and central interval for beta_1 + beta_2."""
    names = _coef_names(p, s)
    i1 = names.index("net_lag1_stage1")
    i2 = names.index("net_lag2_stage1")

    contrast = np.zeros(len(names))
    contrast[i1] = 1.0
    contrast[i2] = 1.0

    mean = float(contrast @ model.mu_n)
    scale2 = float(contrast @ model.theta_post_scale @ contrast)
    scale = np.sqrt(max(scale2, 0.0))
    critical = _st.t.ppf(0.5 + level / 2, df=model.a_n)

    lo = mean - critical * scale
    hi = mean + critical * scale
    posterior_sd = (
        scale * np.sqrt(model.a_n / (model.a_n - 2))
        if model.a_n > 2
        else np.nan
    )
    return mean, lo, hi, posterior_sd


def fit_spec(Y, W, spec, p=2, s=(1, 1)):
    """Fit one retained simulation specification."""
    stage_weights = compute_stage_weights(W, max(s))

    if spec == "raw":
        panel = np.asarray(Y, dtype=float)

    elif spec == "pc1_adjusted":
        panel = np.asarray(Y, dtype=float)
        mu = panel.mean(axis=0, keepdims=True)
        centred = panel - mu
        _, _, Vt = np.linalg.svd(centred, full_matrices=False)
        loading = Vt[0]
        panel = centred - np.outer(centred @ loading, loading)

    else:
        raise ValueError(f"Unknown simulation specification {spec!r}.")

    if not np.all(np.isfinite(panel)) or np.abs(panel).max() > 1e3:
        return np.nan, np.nan, np.nan, np.nan

    model = BayesianGNAR(
        p=p,
        s=list(s),
        prior_type="minnesota",
        lambda1=MINN["lambda1"],
        lambda2=MINN["lambda2"],
        lambda3=MINN["lambda3"],
        rw_centre=MINN["rw_centre"],
    )

    try:
        model.fit(panel, stage_weights)
    except (np.linalg.LinAlgError, FloatingPointError):
        return np.nan, np.nan, np.nan, np.nan

    return _net_sum_interval(model, p, s)


def run_cells(
    dgp_rows,
    specs=SPECS,
    n_reps=N_REPS,
    W=W_blk,
    loadings=None,
    require_complete=True,
):
    """Run retained specifications over a collection of DGP cells."""
    out = []

    for row in dgp_rows:
        for spec in specs:
            estimates = []
            widths = []
            exclusions = 0
            failures = 0

            for rep in range(n_reps):
                # These seeds and the per-specification simulation convention match
                # the established simulation exactly.
                Y = simulate_panel(
                    T_SIM,
                    N_SIM,
                    CAL["a1"],
                    CAL["a2"],
                    row["b1"],
                    row["b2"],
                    row["lam"],
                    W,
                    sig_e=CAL["sig_e"],
                    seed=10_000 + rep,
                    loadings=loadings,
                )

                mean, lo, hi, _ = fit_spec(Y, W, spec)
                if not np.all(np.isfinite([mean, lo, hi])):
                    failures += 1
                    continue

                estimates.append(mean)
                widths.append(hi - lo)
                exclusions += int(lo > 0 or hi < 0)

            used = len(estimates)

            if require_complete and used != n_reps:
                raise RuntimeError(
                    f"{row['label']} / {spec}: "
                    f"{failures} of {n_reps} replications failed."
                )

            truth = row["b1"] + row["b2"]
            rejection_rate = exclusions / used if used else np.nan
            estimate_mean = float(np.mean(estimates)) if estimates else np.nan
            rate_mc_se = (
                float(np.sqrt(rejection_rate * (1 - rejection_rate) / used))
                if used
                else np.nan
            )
            est_mean_mc_se = (
                float(np.std(estimates, ddof=1) / np.sqrt(used))
                if used > 1
                else np.nan
            )

            out.append({
                "dgp": row["label"],
                "spec": spec,
                "truth": truth,
                "n_requested": int(n_reps),
                "n_used": int(used),
                "n_failed": int(failures),
                "rej_rate": rejection_rate,
                "rate_mc_se": rate_mc_se,
                "est_mean": estimate_mean,
                "bias": estimate_mean - truth if used else np.nan,
                "est_mean_mc_se": est_mean_mc_se,
                "mean_width": float(np.mean(widths)) if widths else np.nan,
            })

    return pd.DataFrame(out)


_Y = simulate_panel(
    T_SIM,
    N_SIM,
    CAL["a1"],
    CAL["a2"],
    CAL["b1"],
    CAL["b2"],
    CAL["lam"],
    W_blk,
    seed=0,
)

for spec in SPECS:
    mean, lo, hi, _ = fit_spec(_Y, W_blk, spec)
    print(
        f"{spec:14s} beta1+beta2={mean:+.5f} "
        f"[{lo:+.5f}, {hi:+.5f}] "
        f"(true {CAL['b1'] + CAL['b2']:+.5f})"
    )

raw            beta1+beta2=+0.01979 [+0.01560, +0.02398] (true +0.01500)
pc1_adjusted   beta1+beta2=+0.02173 [+0.01046, +0.03299] (true +0.01500)


## Experiment 1: common-factor confounding

DGP 2 contains the common factor but no network effect. The interval-exclusion frequency therefore measures spurious network detection.

In [5]:
t0 = time.time()

dgp2 = [{
    "label": "DGP2 AR+factor (b=0)",
    "lam": CAL["lam"],
    "b1": 0.0,
    "b2": 0.0,
}]

res2 = run_cells(dgp2)

print(res2.round(5).to_string(index=False))
print(f"runtime={time.time() - t0:.1f}s")

                 dgp         spec  truth  n_requested  n_used  n_failed  rej_rate  rate_mc_se  est_mean     bias  est_mean_mc_se  mean_width
DGP2 AR+factor (b=0)          raw    0.0           10      10         0       1.0         0.0   0.01136  0.01136         0.00111     0.00825
DGP2 AR+factor (b=0) pc1_adjusted    0.0           10      10         0       0.0         0.0  -0.00112 -0.00112         0.00179     0.02540
runtime=0.5s


### Empirical-PC1-share calibration check

The DGP 2 factor strength is separately recalibrated to the empirical covariance-PC1 share. This is a supplementary calibration check rather than the principal DGP 2 result.

In [6]:
LAM_MATCHED, share_matched, matched_ok = match_lam(
    np.ones(N_SIM),
    0.0,
    0.0,
    target=PC1_TARGET,
    W=W_blk,
    strict=True,
)

dgp2_matched = [{
    "label": "DGP2 AR+factor, share matched",
    "lam": LAM_MATCHED,
    "b1": 0.0,
    "b2": 0.0,
}]

res2m = run_cells(dgp2_matched)

print(
    f"DGP2 lam recalibrated {CAL['lam']:.5f} -> {LAM_MATCHED:.5f}; "
    f"realised PC1 share={share_matched:.5f}"
)
print(res2m.round(5).to_string(index=False))

DGP2 lam recalibrated 0.02000 -> 0.03227; realised PC1 share=0.88829
                          dgp         spec  truth  n_requested  n_used  n_failed  rej_rate  rate_mc_se  est_mean     bias  est_mean_mc_se  mean_width
DGP2 AR+factor, share matched          raw    0.0           10      10         0       1.0         0.0   0.01208  0.01208         0.00098     0.00779
DGP2 AR+factor, share matched pc1_adjusted    0.0           10      10         0       0.0         0.0  -0.00166 -0.00166         0.00189     0.02571


## Experiment 2: network-effect recovery and calibration

DGP 3 is the network-only calibration case. DGP 4 combines the common factor with the planted network effect and provides the principal detection/recovery comparison.

In [7]:
t0 = time.time()

dgp34 = [
    {
        "label": "DGP3 AR+net (no factor)",
        "lam": 0.0,
        "b1": CAL["b1"],
        "b2": CAL["b2"],
    },
    {
        "label": "DGP4 AR+factor+net",
        "lam": CAL["lam"],
        "b1": CAL["b1"],
        "b2": CAL["b2"],
    },
]

res34 = run_cells(dgp34)

print(res34.round(5).to_string(index=False))
print(f"runtime={time.time() - t0:.1f}s")

                    dgp         spec  truth  n_requested  n_used  n_failed  rej_rate  rate_mc_se  est_mean     bias  est_mean_mc_se  mean_width
DGP3 AR+net (no factor)          raw  0.015           10      10         0       1.0     0.00000   0.01398 -0.00102         0.00102     0.01147
DGP3 AR+net (no factor) pc1_adjusted  0.015           10      10         0       0.7     0.14491   0.01418 -0.00082         0.00190     0.02308
     DGP4 AR+factor+net          raw  0.015           10      10         0       1.0     0.00000   0.01633  0.00133         0.00081     0.00813
     DGP4 AR+factor+net pc1_adjusted  0.015           10      10         0       0.6     0.15492   0.01293 -0.00207         0.00294     0.02355
runtime=1.3s


## Experiment 3: factor-loading heterogeneity

A single covariance-PCA loading-dispersion path is retained. Factor strength is recalibrated at each point so that changes in loading heterogeneity are not mechanically confounded with changes in the leading-component share.

In [8]:
r6 = load_result("nb6_common_factor")


def unit_mean(v):
    """Rescale a loading vector to have mean one."""
    v = np.asarray(v, dtype=float)
    if np.isclose(v.mean(), 0.0):
        raise ValueError("Loading vector has near-zero mean.")
    return v / v.mean()


def blend(loadings, weight):
    """Interpolate between equal and heterogeneous loadings."""
    return 1.0 + weight * (np.asarray(loadings, dtype=float) - 1.0)


loading_full = unit_mean(r6["pc1_loadings"])
LOADING_SET = "covariance PCA"
W_DISPERSION = [0.0, 0.25, 0.5, 0.75, 1.0] if not QUICK else [0.0, 1.0]

print(
    f"{LOADING_SET}: CV={loading_full.std() / loading_full.mean():.5f}; "
    f"max/min={np.abs(loading_full).max() / np.abs(loading_full).min():.5f}"
)

t0 = time.time()
disp_frames = []
disp_meta = []

for weight in W_DISPERSION:
    loading_vec = blend(loading_full, weight)
    loading_cv = float(loading_vec.std() / loading_vec.mean())

    for b1, b2, dgp_base in [
        (0.0, 0.0, "DGP2"),
        (CAL["b1"], CAL["b2"], "DGP4"),
    ]:
        lam_w, share_w, calibration_ok = match_lam(
            loading_vec,
            b1,
            b2,
            target=PC1_TARGET,
            W=W_blk,
            strict=False,
        )

        label = f"{dgp_base} {LOADING_SET} w={weight:.2f}"

        disp_meta.append({
            "loading_set": LOADING_SET,
            "w": weight,
            "dgp": dgp_base,
            "lam": lam_w,
            "realised_share": share_w,
            "loading_cv": loading_cv,
            "calibration_ok": calibration_ok,
        })

        if not calibration_ok:
            for spec in SPECS:
                disp_frames.append(pd.DataFrame([{
                    "dgp": label,
                    "spec": spec,
                    "truth": b1 + b2,
                    "n_requested": N_REPS,
                    "n_used": 0,
                    "n_failed": N_REPS,
                    "rej_rate": np.nan,
                    "rate_mc_se": np.nan,
                    "est_mean": np.nan,
                    "bias": np.nan,
                    "est_mean_mc_se": np.nan,
                    "mean_width": np.nan,
                    "loading_set": LOADING_SET,
                    "w": weight,
                    "dgp_base": dgp_base,
                    "lam": lam_w,
                    "realised_share": share_w,
                    "loading_cv": loading_cv,
                    "calibration_ok": False,
                }]))
            continue

        result = run_cells(
            [{
                "label": label,
                "lam": lam_w,
                "b1": b1,
                "b2": b2,
            }],
            loadings=loading_vec,
            require_complete=False,
        )

        result["loading_set"] = LOADING_SET
        result["w"] = weight
        result["dgp_base"] = dgp_base
        result["lam"] = lam_w
        result["realised_share"] = share_w
        result["loading_cv"] = loading_cv
        result["calibration_ok"] = True
        disp_frames.append(result)

dispersion = pd.concat(disp_frames, ignore_index=True)

print(f"loading-heterogeneity sensitivity runtime={time.time() - t0:.1f}s")
print(pd.DataFrame(disp_meta).round(5).to_string(index=False))
print(
    dispersion.pivot_table(
        index=["dgp_base", "w"],
        columns="spec",
        values="rej_rate",
    ).round(5).to_string()
)

covariance PCA: CV=1.57773; max/min=88.36721
loading-heterogeneity sensitivity runtime=3.2s
   loading_set   w  dgp      lam  realised_share  loading_cv  calibration_ok
covariance PCA 0.0 DGP2  0.03227         0.88829     0.00000            True
covariance PCA 0.0 DGP4  0.02124         0.88829     0.00000            True
covariance PCA 1.0 DGP2  0.01809         0.88829     1.57773            True
covariance PCA 1.0 DGP4 25.60000         0.87079     1.57773           False
spec          pc1_adjusted  raw
dgp_base w                     
DGP2     0.0           0.0  1.0
         1.0           0.0  0.0
DGP4     0.0           0.6  1.0


## Experiment 4: geographic densification

The empirical geographic network is evaluated at \(k\in\{2,5,12,\mathrm{complete}\}\). Factor strength is recalibrated at each network so that realised common-factor dominance remains approximately comparable. Changing \(k\) changes neighbourhood composition as well as density.

In [9]:
d_emp = load_data(network="geographic")
W_emp = compute_stage_weights(d_emp["W"], 1)[0]

t0 = time.time()
dens_frames = []
dens_meta = []

for k in [2, 5, 12, None]:
    Wk = knn_sparsify(W_emp, k)
    k_label = "complete" if k is None else str(k)
    density = float((Wk > 0).sum() / (N_SIM * (N_SIM - 1)))

    for b1, b2, dgp_base in [
        (0.0, 0.0, "DGP2"),
        (CAL["b1"], CAL["b2"], "DGP4"),
    ]:
        lam_k, share_k, calibration_ok = match_lam(
            np.ones(N_SIM),
            b1,
            b2,
            target=PC1_TARGET,
            W=Wk,
            strict=True,
        )

        result = run_cells(
            [{
                "label": f"{dgp_base} k={k_label}",
                "lam": lam_k,
                "b1": b1,
                "b2": b2,
            }],
            n_reps=DENSITY_REPS,
            W=Wk,
        )

        result["k"] = k_label
        result["density"] = density
        result["dgp_base"] = dgp_base
        result["lam"] = lam_k
        result["realised_share"] = share_k
        result["calibration_ok"] = calibration_ok
        dens_frames.append(result)

        dens_meta.append({
            "k": k_label,
            "density": density,
            "dgp": dgp_base,
            "lam": lam_k,
            "realised_share": share_k,
            "calibration_ok": calibration_ok,
        })

density_ladder = pd.concat(dens_frames, ignore_index=True)

print(f"geographic densification runtime={time.time() - t0:.1f}s")
print(pd.DataFrame(dens_meta).round(5).to_string(index=False))
print(
    density_ladder.pivot_table(
        index=["k", "density"],
        columns=["dgp_base", "spec"],
        values="rej_rate",
    ).round(5).to_string()
)

geographic densification runtime=5.5s
       k  density  dgp     lam  realised_share  calibration_ok
       2  0.09091 DGP2 0.03227         0.88829            True
       2  0.09091 DGP4 0.02218         0.88829            True
       5  0.22727 DGP2 0.03227         0.88829            True
       5  0.22727 DGP4 0.01992         0.88829            True
      12  0.54545 DGP2 0.03227         0.88829            True
      12  0.54545 DGP4 0.02018         0.88829            True
complete  1.00000 DGP2 0.03227         0.88829            True
complete  1.00000 DGP4 0.02015         0.88829            True
dgp_base                  DGP2              DGP4     
spec              pc1_adjusted  raw pc1_adjusted  raw
k        density                                     
12       0.545455          0.0  1.0          0.2  1.0
2        0.090909          0.0  1.0          1.0  1.0
5        0.227273          0.0  1.0          0.8  1.0
complete 1.000000          0.0  1.0          0.0  1.0


## Summary

In [10]:
false_positive = res2.set_index("spec")["rej_rate"]

dgp4 = res34[
    res34["dgp"].str.startswith("DGP4")
].set_index("spec")

verdict = pd.DataFrame({
    "false_positive_DGP2": false_positive,
    "detection_DGP4": dgp4["rej_rate"],
    "bias_DGP4": dgp4["bias"],
})

print(verdict.round(5).to_string())

              false_positive_DGP2  detection_DGP4  bias_DGP4
spec                                                        
raw                           1.0             1.0    0.00133
pc1_adjusted                  0.0             0.6   -0.00207


## Save results

In [ ]:
save_result("nb10_simulation", {
    "calibration": CAL,
    "n_reps": N_REPS,
    "density_n_reps": DENSITY_REPS,
    "T": T_SIM,
    "burn_in": BURN_IN,
    "N": N_SIM,
    "spec_labels": {
        "raw": "Raw",
        "pc1_adjusted": "PC1-adjusted",
    },
    "block_density": block_density,
    "block_graph_proposal_probabilities": {
        "within": 0.7,
        "between": 0.05,
    },
    "full_system_radius": float(rad),
    "domestic_root": float(dom),
    "realised_pc1_shares": realised_shares,
    "pc1_target_from_nb6": PC1_TARGET,
    "lam_matched_dgp2": float(LAM_MATCHED),
    "realised_pc1_share_matched_dgp2": float(share_matched),
    "dgp2_spurious": res2.to_dict(orient="records"),
    "dgp2_spurious_share_matched": res2m.to_dict(orient="records"),
    "dgp34_power": res34.to_dict(orient="records"),
    "dispersion_sweep": dispersion.to_dict(orient="records"),
    "dispersion_meta": disp_meta,
    "dispersion_w_ladder": W_DISPERSION,
    "loading_sets": {
        LOADING_SET: [float(x) for x in loading_full],
    },
    "density_ladder": density_ladder.to_dict(orient="records"),
    "density_ladder_meta": dens_meta,
    "verdict": verdict.reset_index().to_dict(orient="records"),
    "config": run_config(
        p=2,
        stages=[1, 1],
        purpose="controlled_simulation",
    ),
    "quick": QUICK,
})

print("saved nb10_simulation")